Setup env

In [ ]:
pip install nltk gensim scikit-learn numpy seaborn pyspellchecker

In [ ]:
pip install --user --upgrade scipy

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') 
# nltk.download('resource_name')

Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from spellchecker import SpellChecker
from nltk import pos_tag
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer



In [ ]:
def load_tweet_file(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    if '\n' in content.strip() and content.count('\n') > 5:
        tweets = [line.strip() for line in content.split('\n') if line.strip()]
    else:
        tweets = [t.strip() for t in content.split(',') if t.strip()]
    return pd.DataFrame({'tweets': tweets})

neg = load_tweet_file('../data/processedNegative.csv')
pos = load_tweet_file('../data/processedPositive.csv')
neu = load_tweet_file('../data/processedNeutral.csv')

print(neg.shape, pos.shape, neu.shape)


In [ ]:
neg['label'] = 'Negative'
pos['label'] = 'Positive'
neu['label'] = 'Neutral'

df = pd.concat([neg, pos, neu], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['label'].value_counts())

df.head()

df['label'].value_counts().plot(kind='bar')

In [ ]:
df['char_length'] = df['tweets'].apply(len)

df['word_count'] = df['tweets'].apply(lambda x: len(x.split()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['char_length'], bins=50, ax=axes[0])
axes[0].set_title('Distribution longueur (caractères)')
axes[0].set_xlabel('Nombre de caractères')

sns.histplot(df['word_count'], bins=50, ax=axes[1])
axes[1].set_title('Distribution longueur (mots)')
axes[1].set_xlabel('Nombre de mots')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='label', y='word_count')
plt.title('Longueur des tweets par label')
plt.show()

In [ ]:
n_dup = df.duplicated(subset='tweets').sum()

duplicated_tweets = df[df.duplicated(subset='tweets', keep=False)].sort_values('tweets')
duplicated_tweets.head(10)

duplicates_check = df[df.duplicated(subset='tweets', keep=False)]
label_consistency = duplicates_check.groupby('tweets')['label'].nunique()
inconsistent = label_consistency[label_consistency > 1]

In [ ]:
df_clean = df.drop_duplicates(subset='tweets', keep='first')
df_clean['tweets'] = df_clean['tweets'].fillna('')
df_clean = df_clean.dropna(subset=['tweets'])

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['clean_tweets'] = df_clean['tweets'].apply(basic_clean)
df_clean

In [ ]:
def tokenize_only(text):
    tokens = word_tokenize(text)
    return ' '.join(tokens)

df_clean['tokenized'] = df_clean['clean_tweets'].apply(tokenize_only)

In [ ]:
def apply_stemming(text):
    tokens = word_tokenize(text)
    stemmed = [stemmer.stem(t) for t in tokens]
    return ' '.join(stemmed)

df_clean['stemmed'] = df_clean['clean_tweets'].apply(apply_stemming)

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def apply_lemmatization(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(t, get_wordnet_pos(tag)) for t, tag in tagged]
    return ' '.join(lemmatized)

df_clean['lemmatized'] = df_clean['clean_tweets'].apply(apply_lemmatization)

In [ ]:
spell = SpellChecker()

def correct_spelling(text):
    tokens = text.split()
    corrected = []
    for t in tokens:
        c = spell.correction(t)
        corrected.append(c if c is not None else t)
    return ' '.join(corrected)

df_clean['spell_corrected'] = df_clean['clean_tweets'].apply(correct_spelling)

In [ ]:
df_clean['stemmed_misspellings'] = df_clean['spell_corrected'].apply(apply_stemming)
df_clean['lemmatized_misspellings'] = df_clean['spell_corrected'].apply(apply_lemmatization)

In [ ]:
def remove_stopwords(text):
    tokens = text.split()
    filtered = [t for t in tokens if t not in stop_words]
    return ' '.join(filtered)

df_clean['lemmatized_no_stopwords'] = df_clean['lemmatized'].apply(remove_stopwords)

In [ ]:
preprocessing_variants = {
    'tokenization': df_clean['tokenized'],
    'stemming': df_clean['stemmed'],
    'lemmatization': df_clean['lemmatized'],
    'stemming_misspellings': df_clean['stemmed_misspellings'],
    'lemmatization_misspellings': df_clean['lemmatized_misspellings'],
    'lemmatization_no_stopwords': df_clean['lemmatized_no_stopwords'],
}

In [48]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorization_types = ['binary', 'count', 'tfidf']

matrix_table = pd.DataFrame(index=preprocessing_variants.keys(), columns=vectorization_types)

for name, series in preprocessing_variants.items():
    binary_vec = CountVectorizer(binary=True)
    X_binary = binary_vec.fit_transform(series)

    count_vec = CountVectorizer(binary=False)
    X_count = count_vec.fit_transform(series)

    tfidf_vec = TfidfVectorizer()
    X_tfidf = tfidf_vec.fit_transform(series)

    matrix_table.loc[name, 'binary'] = X_binary
    matrix_table.loc[name, 'count'] = X_count
    matrix_table.loc[name, 'tfidf'] = X_tfidf

In [53]:
matrix_table

,binary,count,tfidf
tokenization,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
stemming,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
lemmatization,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
stemming_misspellings,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
lemmatization_misspellings,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
lemmatization_no_stopwords,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...


In [52]:
X = matrix_table.loc['stemming', 'tfidf']

print(X.shape)
print(X.toarray()[:550])

(3451, 5082)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
